# Milestone 1 lab, notebook 5 of 7: Every morning

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and the source | Read the question. Meet the order system. Look at one order |
| 2. Read every order | Follow the pages of the API to the end |
| 3. The answer by hand | Work out the answer with plain Python. Meet the time zone trap |
| 4. Transform, then load | Put the orders in a warehouse table. Ask it one SQL question |
| **5. Every morning** (you are here) | Run it again the next day. Watch an old number change |
| 6. Why batch, and where it breaks | Measure how late the report is. Then watch the load fail |
| 7. The production version | The same logic as a tested program |

So far you answered three parts of the question, for one morning. The fourth part is still open:

> ...and has any earlier day changed since I last looked?

That needs a second morning. This notebook runs the pipeline on Monday, then on Tuesday, and compares.

In [1]:
import duckdb
import pandas as pd
from lab_helpers import LAB_WAREHOUSE, api, arden_time, money, set_morning

## Step 1: The whole pipeline as two small functions

These are the steps of notebooks 2 and 4, put together. Nothing is new.

In [2]:
def extract():
    """Read every order. Returns the orders, the number of calls, and whether the API refused."""
    orders, cursor, calls = [], None, 0
    while True:
        answer = api.get("/orders", params={"cursor": cursor} if cursor else None)
        calls += 1
        if answer.status_code == 429:
            return orders, calls, True
        body = answer.json()
        orders += body["data"]
        cursor = body["next_cursor"]
        if cursor is None:
            return orders, calls, False


def transform(orders):
    """One row per order, with its business day."""
    table = pd.DataFrame(orders)
    table["placed_at"] = pd.to_datetime(table["placed_at"], utc=True)
    table["updated_at"] = pd.to_datetime(table["updated_at"], utc=True)
    table["order_date"] = table["placed_at"].dt.tz_convert("America/Chicago").dt.date
    table["loaded_at"] = pd.Timestamp.now(tz="UTC")  # the moment this pipeline copied the row
    return table[
        [
            "order_id",
            "city",
            "status",
            "placed_at",
            "order_date",
            "total_cents",
            "refunded_cents",
            "updated_at",
            "loaded_at",
        ]
    ]

## Step 2: Monday morning

In [3]:
LAB_WAREHOUSE.unlink(missing_ok=True)
con = duckdb.connect(str(LAB_WAREHOUSE))
con.execute("SET TimeZone = 'UTC'")

set_morning("2026-06-08")
orders, calls, refused = extract()
monday = transform(orders)
con.execute("CREATE TABLE orders AS SELECT * FROM monday")

print(f"Monday: {len(monday)} orders loaded in {calls} calls.")

It is now Monday 2026-06-08 07:00:00 in Arden.
Monday: 226 orders loaded in 10 calls.


## Step 3: Tuesday morning

A day passes. Move the clock, and run the same steps.

Before the new rows replace the table, keep Monday's table under another name. Without it there is nothing to compare with.

In [4]:
set_morning("2026-06-09")
orders, calls, refused = extract()
tuesday = transform(orders)

con.execute("ALTER TABLE orders RENAME TO orders_previous")  # keep Monday's table
con.execute("CREATE TABLE orders AS SELECT * FROM tuesday")

print(f"Tuesday: {len(tuesday)} orders loaded in {calls} calls.")

It is now Tuesday 2026-06-09 07:00:00 in Arden.
Tuesday: 251 orders loaded in 11 calls.


## Step 4: Has an earlier day changed?

Work out the day totals from both tables, side by side. Look only at days Monday's report already showed: up to Sunday 7 June.

In [5]:
DAY_TOTALS = """
    SELECT order_date,
           count(*) FILTER (WHERE status <> 'cancelled') AS orders,
           sum(total_cents - refunded_cents) FILTER (WHERE status <> 'cancelled')::BIGINT AS revenue_cents
    FROM {table} GROUP BY order_date
"""

compare = con.sql(f"""
    SELECT now.order_date,
           old.orders AS orders_on_monday,   now.orders AS orders_on_tuesday,
           old.revenue_cents AS cents_on_monday, now.revenue_cents AS cents_on_tuesday,
           now.revenue_cents - old.revenue_cents AS change_cents
    FROM ({DAY_TOTALS.format(table="orders")}) AS now
    JOIN ({DAY_TOTALS.format(table="orders_previous")}) AS old USING (order_date)
    WHERE now.order_date <= '2026-06-07'
    ORDER BY now.order_date
""").df()
compare

,order_date,orders_on_monday,orders_on_tuesday,cents_on_monday,cents_on_tuesday,change_cents
0,2026-06-01,28,28,73076,73076,0
1,2026-06-02,32,32,75189,75189,0
2,2026-06-03,35,35,83247,83247,0
3,2026-06-04,30,30,59198,59198,0
4,2026-06-05,37,37,97682,97104,-578
5,2026-06-06,33,33,78671,78671,0
6,2026-06-07,30,30,80711,80711,0


Six days did not move. **Friday 5 June did.** It brought in $976.82 when the manager looked on Monday. On Tuesday the same Friday is worth $971.04.

Finance will ask why. Find the order.

In [6]:
changed = con.sql("""
    SELECT now.order_id, now.order_date, now.city,
           old.status AS status_monday, now.status AS status_tuesday,
           old.refunded_cents AS refunded_monday, now.refunded_cents AS refunded_tuesday
    FROM orders AS now
    JOIN orders_previous AS old USING (order_id)
    WHERE now.order_date <= '2026-06-07'
      AND (now.total_cents <> old.total_cents
           OR now.refunded_cents <> old.refunded_cents
           OR (now.status = 'cancelled') <> (old.status = 'cancelled'))
""").df()
changed

,order_id,order_date,city,status_monday,status_tuesday,refunded_monday,refunded_tuesday
0,1130,2026-06-05,NGT,delivered,delivered,0,578


One order: 1130. Ask the order system for its full story.

In [7]:
order = api.get("/orders/1130").json()

print(f"Order {order['order_id']} in {order['city']}")
print("  placed:   ", arden_time(order["placed_at"]).strftime("%A %d %B, %H:%M"))
print("  delivered:", arden_time(order["delivered_at"]).strftime("%A %d %B, %H:%M"))
print(
    "  refunded: ",
    arden_time(order["refunded_at"]).strftime("%A %d %B, %H:%M"),
    "  amount",
    money(order["refunded_cents"]),
)
for item in order["items"]:
    print(f"    {item['quantity']} x {item['product_name']} at {money(item['unit_price_cents'])}")

Order 1130 in NGT
  placed:    Friday 05 June, 10:20
  delivered: Friday 05 June, 15:45
  refunded:  Monday 08 June, 11:00   amount $5.78
    2 x Whole milk 2L at $2.89
    1 x Cheddar 200g at $4.49
    1 x Pasta 500g at $1.89
    1 x Tomato sauce at $2.59


The customer ordered on Friday. The order was delivered on Friday. Three days later, on Monday, the milk was refunded: 2 × $2.89 = $5.78.

The refund belongs to **Friday's** order, so it lowers **Friday's** number.

> **Yesterday's orders are not the same today as they were yesterday.**

This is the heart of the whole project. A report is only right on the morning it was made. A pipeline that loads only *new* orders would never see this refund.

## Step 5: A new word, load time

Every row in the warehouse now carries two different times. Look at order 1130.

In [8]:
con.sql("SELECT order_id, placed_at, updated_at, loaded_at FROM orders WHERE order_id = 1130").df()

,order_id,placed_at,updated_at,loaded_at
0,1130,2026-06-05 15:20:00+00:00,2026-06-08 16:00:00+00:00,2026-11-21 08:53:04.072975+00:00


A new word arrives with it: **load time**.

| Column | Its name | It answers |
|---|---|---|
| `placed_at` | **Event time** | When did the customer order? |
| `loaded_at` | **Load time** | When did this pipeline copy the row? |

`loaded_at` is real: it is the moment you ran the cell above. The order system's clock is pretend, so the two will look far apart here. In a real company both are real.

Never use load time for a business number. If Tuesday's run fails and you run it on Wednesday, load time changes. The day the customer ordered does not.

## What you now know

- A second morning showed that Friday's number went down by $5.78, because of one refund three days later.
- Yesterday's orders are not the same today as they were yesterday. A pipeline must load orders that changed, not only new ones.
- Event time is when it happened. Load time is when the pipeline copied it. Business numbers use event time.

**Say it in an interview:** "An order kept changing after it was placed: a refund arrived three days after delivery and lowered an old day's revenue. I keep two times on every row. Event time says which business day the order belongs to. Load time only says when my pipeline copied it."

Next: [notebook 6](06_why_batch_and_where_it_breaks.ipynb) measures how late this report is, and finds where the pipeline breaks.

In [9]:
con.close()